# C2G-Bench — Workload Traces Deep Dive

The benchmark fuses **three real Alibaba production traces** to model the IT load of a 250 MW hyperscale data center.  
Each trace has a different nature, resolution, and role in the physics engine.

| File | Source | Duration | Timescale | Zone | Role in env |
|---|---|---|---|---|---|
| `batch_v2023.csv` | Alibaba Cluster 2023 (OpenB) | 33 days | 5-min ticks | A (GPU) | `P_flex` — schedulable, DVFS-throttleable |
| `dlrm_v2025.csv` | Alibaba Cluster 2025 (DLRM) | 30 days | 5-min ticks | B (CPU) | `P_base` — rigid inference serving |
| `genai_v2026.csv` | Alibaba 2026 GenAI | 1 day (tiled) | 5-min ticks | A (GPU) | `P_base` — rigid GenAI, spike-prone |

**Key distinction:**  
- `P_flex` (batch) can be deferred — the agent earns SLA penalty only for *backlog*, not for dropping.  
- `P_base` (DLRM + GenAI) is non-preemptible — the data center must serve it regardless of grid conditions.

The three power streams plus BESS compose the total IT load fed into the thermal and electrical physics.

In [ ]:
import sys, os
from pathlib import Path

_root = Path.cwd()
for _c in [_root, *_root.parents]:
    if (_c / "pyproject.toml").exists():
        _root = _c
        break
os.chdir(_root)
sys.path.insert(0, str(_root))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from scipy import stats

sns.set_theme(style="whitegrid", font_scale=1.1)
plt.rcParams["figure.dpi"] = 120
PALETTE = ["#1f77b4", "#ff7f0e", "#2ca02c"]   # batch=blue, dlrm=orange, genai=green

# ── Load raw traces ──
TRACE_DIR = Path("data/processed/workload_traces")
batch = pd.read_csv(TRACE_DIR / "batch_v2023.csv")
dlrm  = pd.read_csv(TRACE_DIR / "dlrm_v2025.csv")
genai = pd.read_csv(TRACE_DIR / "genai_v2026.csv")

# Add wall-clock time axes
batch["time_h"] = batch["tick"] * 5 / 60
dlrm["time_h"]  = dlrm["tick"]  * 5 / 60
genai["time_h"] = genai["tick"] * 5 / 60

print(f"batch_v2023 : {len(batch):,} ticks  = {len(batch)*5/60/24:.0f} days")
print(f"dlrm_v2025  : {len(dlrm):,} ticks  = {len(dlrm)*5/60/24:.0f} days")
print(f"genai_v2026 : {len(genai):,} ticks  = {len(genai)*5/60/24:.1f} day (tiled cyclically in env)")
print("\nAll traces loaded.")

## 1. From Raw Metrics to Power

The `WorkloadOrchestrator` converts each trace's raw counter into a rack-level power estimate using the non-linear server power model:

$$P_{server}(u) = N_{racks} \times \left[ P_{idle} + (P_{max} - P_{idle}) \cdot u^{\alpha} \right]$$

where $\alpha = 1.4$ for GPU racks (superlinear) and $\alpha = 1.2$ for CPU inference racks.

| Stream | Racks | $P_{idle}$ | $P_{max}$ | $\alpha$ | Normaliser |
|---|---|---|---|---|---|
| Batch (Zone A flex) | 1 200 | 8 kW | 25 kW | 1.4 | `gpu_milli_request / 12 620` |
| GenAI (Zone A base) | 800 | 8 kW | 25 kW | 1.4 | `avg_gpu_duty_cycle / 100` |
| DLRM (Zone B base) | 2 500 | 4 kW | 16 kW | 1.2 | `active_gpu_count / 227` |

In [ ]:
# Replicate the physics constants from workload.py
_BATCH_GPU_MILLI_MAX = 12_620.0
_DLRM_GPU_MAX        = 227.0
_GENAI_DUTY_MAX      = 100.0

_ZONE_A_N_RACKS_FLEX = 1_200
_ZONE_A_N_RACKS_BASE = 800
_ZONE_B_N_RACKS      = 2_500
_P_IDLE_A_KW, _P_MAX_A_KW, _ALPHA_A = 8.0, 25.0, 1.4
_P_IDLE_B_KW, _P_MAX_B_KW, _ALPHA_B = 4.0, 16.0, 1.2

def rack_power_kw(u, n_racks, p_idle, p_max, alpha):
    u = np.clip(u, 0.0, 1.0)
    return n_racks * (p_idle + (p_max - p_idle) * u ** alpha)

batch["util"]    = batch["gpu_milli_request"] / _BATCH_GPU_MILLI_MAX
dlrm["util"]     = dlrm["active_gpu_count"]   / _DLRM_GPU_MAX
genai["util"]    = genai["avg_gpu_duty_cycle"] / _GENAI_DUTY_MAX

batch["power_mw"] = rack_power_kw(batch["util"].values, _ZONE_A_N_RACKS_FLEX,
                                  _P_IDLE_A_KW, _P_MAX_A_KW, _ALPHA_A) / 1e3
dlrm["power_mw"]  = rack_power_kw(dlrm["util"].values,  _ZONE_B_N_RACKS,
                                  _P_IDLE_B_KW, _P_MAX_B_KW, _ALPHA_B) / 1e3
genai["power_mw"] = rack_power_kw(genai["util"].values, _ZONE_A_N_RACKS_BASE,
                                  _P_IDLE_A_KW, _P_MAX_A_KW, _ALPHA_A) / 1e3

for name, df in [("batch_v2023", batch), ("dlrm_v2025", dlrm), ("genai_v2026", genai)]:
    p = df["power_mw"]
    idle = rack_power_kw(0, {"batch_v2023":_ZONE_A_N_RACKS_FLEX,"dlrm_v2025":_ZONE_B_N_RACKS,"genai_v2026":_ZONE_A_N_RACKS_BASE}[name],
                         {"batch_v2023":_P_IDLE_A_KW,"dlrm_v2025":_P_IDLE_B_KW,"genai_v2026":_P_IDLE_A_KW}[name],
                         {"batch_v2023":_P_MAX_A_KW, "dlrm_v2025":_P_MAX_B_KW, "genai_v2026":_P_MAX_A_KW}[name],
                         {"batch_v2023":_ALPHA_A,"dlrm_v2025":_ALPHA_B,"genai_v2026":_ALPHA_A}[name]) / 1e3
    print(f"{name}: P_idle={idle:.1f} MW  P_mean={p.mean():.1f} MW  P_max={p.max():.1f} MW")

## 2. Full Time-Series — 30 Days

Each trace shown at full resolution to reveal its temporal character.

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(16, 9), sharex=False)

datasets = [
    (batch, "gpu_milli_request", "GPU Milli-Request", "batch_v2023  (Zone A, P_flex)", PALETTE[0]),
    (dlrm,  "active_gpu_count",  "Active GPU Count",  "dlrm_v2025  (Zone B, P_base)", PALETTE[1]),
    (genai, "avg_gpu_duty_cycle","GPU Duty Cycle (%)", "genai_v2026  (Zone A, P_base — 1 day tiled)", PALETTE[2]),
]

for ax, (df, col, ylabel, title, color) in zip(axes, datasets):
    ax.plot(df["time_h"], df[col], color=color, lw=0.6, alpha=0.9)
    ax.set_ylabel(ylabel, fontsize=9)
    ax.set_title(title, fontsize=10, loc="left", pad=4)
    ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"Day {int(x//24)+1}"))
    ax.xaxis.set_major_locator(mticker.MultipleLocator(24*7))

axes[-1].set_xlabel("Time")
fig.suptitle("Workload Traces — Full Duration (5-minute resolution)", fontsize=13, y=1.01)
fig.tight_layout()
fig.savefig("fig_workload_traces_full.png", dpi=150, bbox_inches="tight")
plt.show()


## 3. Three-Day Zoom — Diurnal Patterns

Zooming into 72 hours reveals the diurnal structure of each trace.

In [ ]:
ZOOM_H = 72   # hours to zoom

fig, axes = plt.subplots(3, 1, figsize=(14, 8), sharex=True)

for ax, (df, col, ylabel, title, color) in zip(axes, datasets):
    mask = df["time_h"] <= ZOOM_H
    ax.plot(df.loc[mask, "time_h"], df.loc[mask, col], color=color, lw=1.0)
    ax.set_ylabel(ylabel, fontsize=9)
    ax.set_title(title, fontsize=10, loc="left")
    ax.axvspan(0, 24, alpha=0.04, color="gray")
    ax.axvspan(48, 72, alpha=0.04, color="gray")

axes[-1].set_xlabel("Elapsed Hours")
fig.suptitle("Workload Traces — First 72 Hours (diurnal zoom)", fontsize=13, y=1.01)
fig.tight_layout()
fig.savefig("fig_workload_diurnal_zoom.png", dpi=150, bbox_inches="tight")
plt.show()


## 4. Translated Power Streams (MW)

After applying the nonlinear server power model the three utilisation signals become power streams that directly drive the thermal physics.

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(14, 7))

ZOOM_H2 = 7 * 24  # 1 week

# Full duration aligned to 30 days
n_ticks = 8640
t_full  = np.arange(n_ticks) * 5 / 60

batch_p_full  = np.interp(np.arange(n_ticks), batch["tick"], batch["power_mw"])
dlrm_p_full   = np.interp(np.arange(n_ticks), dlrm["tick"],  dlrm["power_mw"])
genai_rep      = np.tile(genai["power_mw"].values, int(np.ceil(n_ticks / len(genai))))[:n_ticks]
p_base_full    = dlrm_p_full + genai_rep
p_total_full   = batch_p_full + p_base_full

# Full 30-day stacked area
axes[0].stackplot(t_full,
    genai_rep, dlrm_p_full, batch_p_full,
    labels=["GenAI P_base (Zone A)", "DLRM P_base (Zone B)", "Batch P_flex (Zone A)"],
    colors=[PALETTE[2], PALETTE[1], PALETTE[0]], alpha=0.75)
axes[0].set_ylabel("Power (MW)")
axes[0].set_title("Stacked IT Power — 30 Days", fontsize=10)
axes[0].legend(loc="upper right", fontsize=8)
axes[0].xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"Day {int(x//24)+1}"))
axes[0].xaxis.set_major_locator(mticker.MultipleLocator(24*7))

# 1-week zoom
mask_w = t_full <= ZOOM_H2
axes[1].stackplot(t_full[mask_w],
    genai_rep[mask_w], dlrm_p_full[mask_w], batch_p_full[mask_w],
    labels=["GenAI P_base", "DLRM P_base", "Batch P_flex"],
    colors=[PALETTE[2], PALETTE[1], PALETTE[0]], alpha=0.75)
axes[1].set_ylabel("Power (MW)")
axes[1].set_xlabel("Time")
axes[1].set_title("Stacked IT Power — First Week (zoom)", fontsize=10)
axes[1].legend(loc="upper right", fontsize=8)

fig.suptitle("IT Power Breakdown: P_base (rigid) vs P_flex (schedulable)", fontsize=13, y=1.01)
fig.tight_layout()
fig.savefig("fig_workload_power_breakdown.png", dpi=150, bbox_inches="tight")
plt.show()

rigid_pct = p_base_full.mean() / p_total_full.mean() * 100
flex_pct  = batch_p_full.mean() / p_total_full.mean() * 100
print(f"Mean total IT power : {p_total_full.mean():.1f} MW")
print(f"  P_base (rigid)    : {p_base_full.mean():.1f} MW  ({rigid_pct:.0f}%)")
print(f"  P_flex (batch)    : {batch_p_full.mean():.1f} MW  ({flex_pct:.0f}%)")


## 5. Utilisation Distributions

Histograms comparing the statistical character of each trace.
- **Batch** is highly bursty (77 % zero arrivals) — scheduling headroom exists most of the time.
- **DLRM** is near-Gaussian and always on — classic two-shift inference pattern.
- **GenAI** is low-duty multimodal — most time at near-zero, then occasional bursts.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

hist_data = [
    (batch["util"], "Batch Utilisation", PALETTE[0]),
    (dlrm["util"],  "DLRM Utilisation",  PALETTE[1]),
    (genai["util"], "GenAI Utilisation", PALETTE[2]),
]

for ax, (data, label, color) in zip(axes, hist_data):
    ax.hist(data, bins=60, color=color, edgecolor="white", linewidth=0.3, density=True, alpha=0.85)
    ax.axvline(data.mean(),   color="black", ls="--", lw=1.5, label=f"Mean={data.mean():.3f}")
    ax.axvline(data.median(), color="gray",  ls=":",  lw=1.5, label=f"Median={data.median():.3f}")
    zeros_pct = (data == 0).mean() * 100
    ax.set_title(f"{label}\n(zeros: {zeros_pct:.0f}%)", fontsize=10)
    ax.set_xlabel("Utilisation [0, 1]")
    ax.set_ylabel("Density")
    ax.legend(fontsize=8)

fig.suptitle("Workload Utilisation Distributions", fontsize=13, y=1.02)
fig.tight_layout()
fig.savefig("fig_workload_distributions.png", dpi=150, bbox_inches="tight")
plt.show()


## 6. Autocorrelation — Temporal Persistence

How long does each trace stay correlated? High autocorrelation means slow-changing load (easier to predict); low means bursty/unpredictable.  
The batch trace decorrelates fastest, making it hardest for the agent to anticipate queue build-up.

In [ ]:
def manual_acf(x, nlags=288):
    x = np.asarray(x, dtype=float)
    x = x - x.mean()
    var = np.var(x)
    if var == 0:
        return np.zeros(nlags + 1)
    acf_vals = [1.0]
    for lag in range(1, nlags + 1):
        acf_vals.append(np.mean(x[lag:] * x[:-lag]) / var)
    return np.array(acf_vals)

lags = np.arange(289)  # 0..288 ticks = 0..24 hours
acf_data = [
    (batch["util"],                 "Batch (P_flex)", PALETTE[0]),
    (dlrm["util"],                  "DLRM (P_base B)", PALETTE[1]),
    (genai["avg_gpu_duty_cycle"],   "GenAI (P_base A)", PALETTE[2]),
]

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
ci = 1.96 / np.sqrt(min(len(batch), len(dlrm), len(genai)))

for ax, (series, title, color) in zip(axes, acf_data):
    acf_vals = manual_acf(series.values, nlags=288)
    ax.plot(lags, acf_vals, color=color, lw=1.2)
    ax.axhline(ci,  color="gray", ls="--", lw=0.8)
    ax.axhline(-ci, color="gray", ls="--", lw=0.8)
    ax.axhline(0,   color="black", lw=0.5)
    ax.fill_between(lags, -ci, ci, alpha=0.1, color="gray")
    ax.set_title(title)
    ax.set_xlabel("Lag (5-min ticks)")
    ax.set_ylabel("ACF")
    ax.set_ylim(-0.3, 1.05)
    # Annotate 1-hour and 1-day decorrelation
    for lag_h, label in [(12, "1 hr"), (288, "24 hr")]:
        if lag_h <= 288:
            ax.axvline(lag_h, color="red", ls=":", lw=0.8, alpha=0.6)
            ax.text(lag_h + 3, 0.9, label, fontsize=7, color="red")

fig.suptitle("Autocorrelation Function (up to 24 h = 288 × 5-min ticks)", fontsize=13, y=1.02)
fig.tight_layout()
fig.savefig("fig_workload_acf.png", dpi=150, bbox_inches="tight")
plt.show()


## 7. GenAI Spike Detection

FERC RegD spikes (duty cycle > P75 = 12.19 %) are flagged as `is_spike=True` in the observation vector (`obs[9]`).  
Spikes trigger a 10% SLA penalty bonus in the reward if the agent successfully handles the load.  
The plot below shows spike frequency throughout the diurnal cycle.

In [ ]:
GENAI_SPIKE_PCT75 = 12.19
genai["is_spike"] = genai["avg_gpu_duty_cycle"] > GENAI_SPIKE_PCT75
genai["hour_of_day"] = (genai["tick"] * 5 / 60) % 24

spike_by_hour = genai.groupby(genai["hour_of_day"].apply(lambda x: int(x)))["is_spike"].mean()

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# Left: duty cycle with spikes highlighted
axes[0].plot(genai["time_h"], genai["avg_gpu_duty_cycle"], color=PALETTE[2], lw=0.8, alpha=0.7)
axes[0].axhline(GENAI_SPIKE_PCT75, color="red", ls="--", lw=1.5,
                label=f"Spike threshold (P75 = {GENAI_SPIKE_PCT75}%)")
spike_mask = genai["is_spike"]
axes[0].scatter(genai.loc[spike_mask, "time_h"], genai.loc[spike_mask, "avg_gpu_duty_cycle"],
                color="red", s=8, zorder=5, label="Spike tick")
axes[0].set_xlabel("Elapsed Hours")
axes[0].set_ylabel("GPU Duty Cycle (%)")
axes[0].set_title("GenAI Duty Cycle — 1-day trace (tiled)", fontsize=10)
axes[0].legend(fontsize=8)

# Right: spike probability by hour
axes[1].bar(spike_by_hour.index, spike_by_hour.values * 100,
            color=PALETTE[2], edgecolor="k", linewidth=0.4)
axes[1].set_xlabel("Hour of Day")
axes[1].set_ylabel("Spike Probability (%)")
axes[1].set_title("Spike Frequency by Hour of Day", fontsize=10)
axes[1].set_xticks(range(0, 24, 2))

fig.suptitle("GenAI v2026 — Spike Analysis (obs[9]: is_spike flag)", fontsize=13, y=1.02)
fig.tight_layout()
fig.savefig("fig_workload_genai_spikes.png", dpi=150, bbox_inches="tight")
plt.show()

print(f"Overall spike rate: {genai['is_spike'].mean()*100:.1f}% of ticks")


## 8. Batch Queue Dynamics (Little's Law)

The `WorkloadOrchestrator` implements a FIFO queue via Little's Law.  
Batch jobs that arrive when `throttle < 1.0` accumulate as backlog (kW equivalent).  
This section simulates the queue under three throttle regimes to show how quickly backlog grows.

In [ ]:
# Simulate queue dynamics for batch trace (first 7 days = 2016 ticks)
N_SIM = 2016
dt_s  = 300.0   # 5-minute trace ticks
_P_FLEX_MAX_KW = rack_power_kw(1.0, _ZONE_A_N_RACKS_FLEX,
                                _P_IDLE_A_KW, _P_MAX_A_KW, _ALPHA_A)

arrivals_kw = batch["power_mw"].values[:N_SIM] * 1e3   # convert MW → kW

def sim_queue(arrivals_kw, throttle):
    backlog = 0.0
    backlogs = []
    capacity_kw = _P_FLEX_MAX_KW * throttle
    for arr in arrivals_kw:
        backlog += arr              # new work arrives
        served = min(backlog, capacity_kw)
        backlog = max(0.0, backlog - served)
        backlogs.append(backlog)
    return np.array(backlogs)

throttles = {"Full (1.0)": 1.0, "Half (0.5)": 0.5, "Off (0.0)": 0.0}
t_days = np.arange(N_SIM) * 5 / 60 / 24

fig, ax = plt.subplots(figsize=(13, 4))
for label, thr in throttles.items():
    bl = sim_queue(arrivals_kw, thr)
    ax.plot(t_days, bl / 1e3, label=f"throttle={label}")

ax.set_xlabel("Day")
ax.set_ylabel("Backlog (MW equivalent)")
ax.set_title("Batch Queue Backlog Under Different Throttle Policies (first 7 days)", fontsize=10)
ax.legend(fontsize=9)
fig.tight_layout()
fig.savefig("fig_workload_queue_dynamics.png", dpi=150, bbox_inches="tight")
plt.show()

print(f"P_flex_max (full throttle): {_P_FLEX_MAX_KW/1e3:.1f} MW")
print(f"Mean arrival rate:          {arrivals_kw.mean()/1e3:.1f} MW")
print(f"Peak arrival:               {arrivals_kw.max()/1e3:.1f} MW")


## 9. Summary Statistics Table

In [ ]:
rows = []
for name, df, col, unit, norm in [
    ("batch_v2023", batch, "gpu_milli_request", "gpu_milli", _BATCH_GPU_MILLI_MAX),
    ("dlrm_v2025",  dlrm,  "active_gpu_count",  "GPUs",      _DLRM_GPU_MAX),
    ("genai_v2026", genai, "avg_gpu_duty_cycle", "%",         _GENAI_DUTY_MAX),
]:
    v = df[col]
    rows.append({
        "Trace": name,
        "Raw metric": col,
        "Duration (days)": f"{len(df)*5/60/24:.1f}",
        "Min": f"{v.min():.2f} {unit}",
        "Mean": f"{v.mean():.2f} {unit}",
        "Max": f"{v.max():.2f} {unit}",
        "Zero ticks (%)": f"{(v==0).mean()*100:.1f}",
        "Mean power (MW)": f"{df['power_mw'].mean():.1f}",
        "Max power (MW)": f"{df['power_mw'].max():.1f}",
    })

display(pd.DataFrame(rows).set_index("Trace"))

## 10. Key Takeaways for Agent Design

1. **Batch (P_flex) is the primary control lever.** Its high zero-fraction (78%) means the data center has
   headroom most of the time — the DVFS throttle action (`action[0]`) can reduce batch demand without
   thermal risk during high-regulation periods.

2. **DLRM (P_base Zone B) is the dominant rigid consumer** (~10 MW mean, Zone B CPU racks).
   The agent cannot shed this load; the only handle is HVAC (`action[2]`) to manage Zone B temperature.

3. **GenAI is bursty and diurnal** — the 1-day trace is tiled cyclically, so the agent sees a predictable
   24-hour pattern. Spikes (`obs[9]`) are concentrated in afternoon hours and trigger the SLA bonus.

4. **Queue builds quickly under throttling.** Even 50% throttle leads to sustained MW-scale backlog
   within a few hours — the agent must balance grid regulation revenue against SLA cost.

5. **All three traces are at 5-minute resolution**, upsampled 60× to 5-second fast-env ticks.
   The workload appears constant within each 5-minute window.